# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/en/latest/) library, following the Croissant metadata schema.

### Dataset Source
The dataset source is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print summary
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published {getattr(metadata, 'datePublished', 'N/A')} | Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}\n")
print(f"Keywords: {', '.join(getattr(metadata, 'keywords', []))}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
# Access record sets via 'record_sets' property
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets were found in the dataset metadata.")
else:
    print(f"{len(record_sets)} record set(s) found:")
    for rs in record_sets:
        # Each record set has an @id, name, description, and fields
        print(f"- Record set @id: {rs.id}")
        print(f"  Name: {getattr(rs, 'name', '')}")
        print(f"  Description: {getattr(rs, 'description', '')}")
        print(f"  Fields:")
        for field in getattr(rs, 'fields', []):
            print(f"    - Field @id: {field.id}; name: {getattr(field, 'name', '')}; data type: {getattr(field, 'data_type', '')}")
        print()

## 3. Data Extraction
Load data from each record set into Pandas DataFrames for further analysis. All loading references are by record set and field `@id` (not names).

In [ ]:
# Extract data for each record set
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

if not record_set_ids:
    print("No record sets available to load records.")
else:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))  # Each record is a dict keyed by field @id
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"DataFrame for record set @id {record_set_id}")
        print(df.columns.tolist())
        print(df.head(), '\n')

# For demonstration, select the first record set and show details if exists
if record_set_ids:
    selected_rs = record_set_ids[0]
    print(f"Selected record set for detailed exploration: {selected_rs}")
    df = dataframes[selected_rs]
    print("Sample rows:")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply standard data preparation: filtering by numeric value (removing outliers), normalization, and grouping. All fields referenced by their `@id`.

*If there are no record sets or fields, the steps will illustrate the workflow for when field IDs become available.*

In [ ]:
# Ensure at least one record set with a numeric field is available
import numpy as np

# Example workflow: adapt field/column @ids as available from your metadata/overview
if dataframes:
    # Select first DataFrame & infer numeric fields
    sample_rs_id = list(dataframes)[0]
    df = dataframes[sample_rs_id]
    
    numeric_id_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_id_candidates:
        numeric_field_id = numeric_id_candidates[0]  # Use the first numeric field (customize for your analysis)
        print(f"Using numeric field @id: {numeric_field_id}")

        # Apply filtering (remove outliers, e.g., keep values within 3 standard deviations)
        mean_val = df[numeric_field_id].mean()
        std_val = df[numeric_field_id].std()
        threshold_hi = mean_val + 3 * std_val
        threshold_lo = mean_val - 3 * std_val

        filtered_df = df[(df[numeric_field_id] > threshold_lo) & (df[numeric_field_id] < threshold_hi)]
        print(f"Filtered records within 3 stddev of mean for {numeric_field_id}:")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize
        filtered_df[numeric_field_id + "_normalized"] = (filtered_df[numeric_field_id] - mean_val) / std_val
        print(f"\nNormalized {numeric_field_id} (mean=0, std=1):")
        print(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

        # Attempt grouping by another column @id (categorical)
        category_candidates = [col for col in df.columns if col != numeric_field_id and df[col].nunique() < df.shape[0] / 2]
        if category_candidates:
            group_field_id = category_candidates[0]
            print(f"\nGrouping by field @id: {group_field_id}")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'std', 'count'])
            print(grouped.head())
        else:
            print("No suitable categorical/group field found for grouping.")
    else:
        print("No numeric field detected for EDA in first record set.")
else:
    print("No DataFrame extracted. EDA steps require field/column ids.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

*You may need to install seaborn for more advanced plots: `!pip install seaborn`*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization of numeric field (adjust field @ids as needed)
if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(6, 4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of field @id: {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot grouped by a categorical variable, if found
    if 'group_field_id' in locals():
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we have:

- Loaded the FAIR² dataset from Croissant metadata using mlcroissant
- Explored record sets and fields by their `@id`
- Extracted data to DataFrames and performed basic EDA including filtering, normalization, grouping, and plotting distributions

This approach can be extended for in-depth domain analysis, model building, or integration with other datasets. Remember to always reference dataset elements by their `@id` as per the Croissant metadata model for reliable, reproducible science.